# Practical AI Inference: From API to Serverless

This notebook is an optional way to follow the workshop. It covers the same concepts and code as the scripts in this repository, so you can run everything from one place in Jupyter or Google Colab.

You'll:

1. Call a hosted model, **GLM-5.3**, on a Runpod Public Endpoint (the API).
2. Deploy your own model, **OpenJev**, to a Runpod Serverless endpoint with one agent prompt, then send it requests.

## Key concepts

**Inference** is using a trained model to produce an output: you send it an input (a prompt, a review, an image) and it returns a result (an answer, a score, a picture). Training builds the model; inference is everything you do with it afterward, and it's what you pay for each time your app calls a model.

Runpod gives you three ways to run inference, from least to most control:

| | Public Endpoint | Serverless | Pod |
| --- | --- | --- | --- |
| **What it is** | A model Runpod already hosts. You call it with your API key. | Your own container on autoscaling GPU workers behind a URL. Workers run only while there's work. | A GPU or CPU machine you rent and manage directly: start, stop, SSH in, expose ports. |
| **You deploy** | Nothing | A container image and endpoint settings | A container image |
| **You pay** | Per output (for example, per million tokens) | Per second while a worker runs, including cold starts | Per hour while the pod runs, plus storage while stopped |
| **Good for** | Trying popular models fast | Your own or custom models with bursty traffic | Development, experiments, and always-on apps |
| **In this workshop** | Part 1: GLM-5.3 | Part 2: OpenJev | Hollow Manor's web server |

A Serverless endpoint comes in two types, and you choose when you create it:

- **Queue**: requests become jobs. You send them to `/run` or `/runsync` and check `/status`, and jobs wait in a queue when workers are busy. Public Endpoints use this same job API.
- **Load balancer**: requests go straight to the worker's own HTTP server, like `/v1/systemone` in Part 2. There's no queue, so requests are dropped when every worker is busy.

## Setup

Install the Python packages, then enter your [Runpod API key](https://docs.runpod.io/get-started/credentials). `getpass` keeps the key out of the notebook, so it isn't saved if you share the file.

In [ ]:
%pip install --quiet openai requests

In [ ]:
import os
from getpass import getpass

if not os.environ.get("RUNPOD_API_KEY"):
    os.environ["RUNPOD_API_KEY"] = getpass("Runpod API key: ")

## Demo: Murder at Hollow Manor

[Murder at Hollow Manor](https://github.com/JessicaGarson/hollow-manor-mystery) is a Halloween murder mystery where every suspect is an LLM served through a Runpod API. [Play it here](https://vq9kfsopdoe2m2-8666.proxy.runpod.net).

## Part 1: From API (GLM-5.3 on a Public Endpoint)

GLM-5.3 is a Z.ai model that's strong at writing code and long, multi-step tasks. It runs on Runpod's Public Endpoints, so there's nothing to deploy and you pay per token: $0.77 per 1M input tokens ($0.143 cached) and $2.42 per 1M output tokens.

### With curl: submit a job, then poll

The request goes to `/run`, which accepts the job right away and returns an ID. The script then polls `/status/{job_id}` every two seconds until the job is `COMPLETED`, and prints the `output`.

The request uses OpenAI-style `messages`. `sampling_params` sets `max_tokens` and `temperature`, and `reasoning_effort` controls how much the model thinks before answering. `low` keeps responses fast.

This cell needs [jq](https://jqlang.org/). On Colab, install it first with `!apt-get -qq install jq`.

In [ ]:
%%bash
# Submit a GLM-5.3 job to the Runpod Public Endpoint, then poll until it finishes.
# Needs RUNPOD_API_KEY and jq.
BASE_URL="https://api.runpod.ai/v2/b1dd8hy2hqo1nb"

RUN_RESPONSE=$(curl --silent --show-error --fail \
  --request POST \
  --url "$BASE_URL/run" \
  --header "Content-Type: application/json" \
  --header "Authorization: Bearer $RUNPOD_API_KEY" \
  --data '{
    "input": {
      "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Runpod?"}
      ],
      "sampling_params": {
        "max_tokens": 4096,
        "temperature": 1
      },
      "model": "glm-5.3",
      "reasoning_effort": "low"
    }
  }')

JOB_ID=$(printf '%s' "$RUN_RESPONSE" | jq --raw-output '.id // empty')

if [ -z "$JOB_ID" ]; then
  printf 'The job submission did not return an ID:\n%s\n' "$RUN_RESPONSE" >&2
  exit 1
fi

printf 'Job submitted: %s\n' "$JOB_ID"

while true; do
  STATUS_RESPONSE=$(curl --silent --show-error --fail \
    --request GET \
    --url "$BASE_URL/status/$JOB_ID" \
    --header "Authorization: Bearer $RUNPOD_API_KEY")

  STATUS=$(printf '%s' "$STATUS_RESPONSE" | jq --raw-output '.status')

  case "$STATUS" in
    COMPLETED)
      printf '%s' "$STATUS_RESPONSE" | jq '.output'
      break
      ;;
    FAILED|CANCELLED|TIMED_OUT)
      printf 'Job ended with status %s:\n%s\n' \
        "$STATUS" "$STATUS_RESPONSE" >&2
      exit 1
      ;;
    *)
      printf 'Status: %s. Waiting...\n' "$STATUS"
      sleep 2
      ;;
  esac
done

### With Python: the OpenAI SDK

The endpoint is OpenAI-compatible, so the OpenAI SDK skips the submit-and-poll step and returns the completion in one call. The same base URL works with any tool that accepts a custom OpenAI base URL.

In [ ]:
import os

from openai import OpenAI

client = OpenAI(
    base_url="https://api.runpod.ai/v2/b1dd8hy2hqo1nb/openai/v1",
    api_key=os.environ["RUNPOD_API_KEY"],
)

completion = client.chat.completions.create(
    model="glm-5.3",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Runpod?"},
    ],
    max_tokens=4096,
    temperature=1,
    reasoning_effort="low",
)

print(completion.choices[0].message.content)

### Try it: stream the reply

To see tokens as they're generated instead of waiting for the full reply, add `stream=True` and iterate over the chunks.

In [ ]:
stream = client.chat.completions.create(
    model="glm-5.3",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Runpod?"},
    ],
    max_tokens=4096,
    temperature=1,
    reasoning_effort="low",
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

## Part 2: To Serverless (OpenJev on your own endpoint)

[OpenJev](https://huggingface.co/openjev/openjev) doesn't chat. It answers typed questions about text (yes/no, multiple choice, or a score) and returns probabilities your code can use.

### Deploy with one prompt

Connect your AI agent (for example, Claude Code) to the [Runpod MCP server](https://github.com/runpod/runpod-mcp), then paste this prompt:

```text
Deploy OpenJev on Runpod Serverless using the image ghcr.io/brandonbondig/openjev-worker@sha256:1f95a727a3de6d0a29808f59d89b49424f4d4bc78c871f5080052abb4105424d (use this exact digest, not the latest tag). Create a load-balancer endpoint (not queue-based) named openjev with:

- One H100 80 GB per worker (OpenJev's measured setup is one 80 GB GPU with native FP8, so no A100 fallback). Look up the H100 GPU pool ID from the Runpod catalog rather than guessing it.
- Environment variables PORT=3000, PORT_HEALTH=3001 and HEALTH_CHECK_PATH=/
- A 60 GB container disk
- Expose HTTP ports 3000 and 3001 in the container configuration
- Active workers 0, maximum workers 1, and an idle timeout of 300 seconds
- Request count scaling (load balancer endpoints don't support queue delay scaling)
- FlashBoot on (the API defaults it to off, unlike the console)

Tell me the hourly price before you create anything. When it's done, give me the endpoint URL and a curl command that sends a test request to `/v1/systemone`.
```

The prompt creates a load-balancer endpoint on one H100 80 GB GPU with 0 active workers, so you pay only while a worker is running. The first request after deployment triggers a cold start: the worker downloads and loads the model, which takes several minutes and is billed.

When the agent finishes, enter the endpoint ID it gives you:

In [ ]:
os.environ["ENDPOINT_ID"] = input("Endpoint ID: ").strip()

### With curl: one yes/no question

This waits for a worker to be ready, then asks whether a reviewer likes the sound quality. `noul` in the response is the probability that the answer is yes.

While waiting, `no workers available` is normal: the worker is still starting. `not allowed for QB API` means the endpoint was created as Queue instead of Load balancer, so create a new one with the correct type.

In [ ]:
%%bash
# Wait for a worker, then send one yes/no question. Needs RUNPOD_API_KEY and ENDPOINT_ID.
ready=0
deadline=$(( $(date +%s) + 1200 ))
while [ "$(date +%s)" -lt "$deadline" ]; do
  resp=$(curl -s -m 130 -w '\n%{http_code}' \
    -H "Authorization: Bearer $RUNPOD_API_KEY" \
    "https://$ENDPOINT_ID.api.runpod.ai/v1/version")
  code=${resp##*$'\n'}
  body=${resp%$'\n'*}
  if [ "$code" = "200" ]; then ready=1; echo "Worker ready."; break; fi
  if [ "$code" = "401" ] || [ "$code" = "403" ]; then echo "HTTP $code: check RUNPOD_API_KEY and its endpoint permissions."; break; fi
  echo "Not ready yet (HTTP $code): ${body:-(no response)}"
  sleep 10
done
[ "$ready" = "1" ] || { echo "Worker not ready. Check the messages above and the endpoint's logs before sending requests."; exit 1; }

curl -sS -m 60 https://$ENDPOINT_ID.api.runpod.ai/v1/systemone \
  -H "Authorization: Bearer $RUNPOD_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{"model":"openjev","state":"Review: The sound is great, but the battery barely lasts a day.","questions":{"likes_sound":{"type":"noul","instructions":"Does the reviewer like the sound quality?"}}}' \
  -w "\n"

### With Python: three typed questions at once

This sends one headphone review with three questions:

- `sentiment` is a **choice** among positive, mixed, and negative.
- `recommends` is a **noul** (yes/no) question.
- `rating` is a **score** on a 1 to 5 star scale.

In [ ]:
import os
import sys
import time

import requests

API_KEY = os.environ.get("RUNPOD_API_KEY")
ENDPOINT_ID = os.environ.get("ENDPOINT_ID")
if not API_KEY or not ENDPOINT_ID:
    sys.exit("Set RUNPOD_API_KEY and ENDPOINT_ID first.")

BASE_URL = f"https://{ENDPOINT_ID}.api.runpod.ai"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}


def wait_until_ready(max_wait=1200):
    """Poll /v1/version until a worker is ready to take requests."""
    deadline = time.time() + max_wait
    while time.time() < deadline:
        try:
            # The worker only answers /v1/version once the model is loaded.
            # If no worker is ready, Runpod returns an error after about
            # 2 minutes, so each check is short and we keep polling.
            r = requests.get(f"{BASE_URL}/v1/version", headers=HEADERS, timeout=130)
            if r.status_code == 200:
                print("Worker ready.")
                return
            if r.status_code in (401, 403):
                sys.exit(f"HTTP {r.status_code}: check RUNPOD_API_KEY and its endpoint permissions.")
            print(f"Not ready yet (HTTP {r.status_code}): {r.text[:200]}")
        except requests.RequestException as e:
            print(f"Not ready yet: {e}")
        time.sleep(10)
    sys.exit("No worker became ready. Check the endpoint's logs in the console.")


def ask(state, questions, attempts=3):
    """Send text plus typed questions to OpenJev and return the answers."""
    body = {"model": "openjev", "state": state, "questions": questions}
    for attempt in range(1, attempts + 1):
        try:
            r = requests.post(
                f"{BASE_URL}/v1/systemone", headers=HEADERS, json=body, timeout=60
            )
            r.raise_for_status()
            return r.json()["answers"]
        except requests.RequestException as e:
            # Covers timeouts, HTTP errors, and dropped connections. If the
            # model server fails, it closes the connection with no JSON body,
            # which raises ConnectionError rather than Timeout.
            print(f"Attempt {attempt} failed: {e}")
            if attempt < attempts:
                time.sleep(10)
    sys.exit(f"No answer after {attempts} attempts. Check the endpoint's logs.")


review = (
    "Review: I bought these headphones for my commute. The sound is great "
    "and the noise cancelling works well on the train, but the battery "
    "barely lasts a day and they hurt my ears after an hour."
)

questions = {
    "sentiment": {
        "type": "choice",
        "instructions": "What is the overall tone of this review?",
        "criteria": {"positive": None, "mixed": None, "negative": None},
    },
    "recommends": {
        "type": "noul",
        "instructions": "Does the reviewer say they would recommend this product?",
    },
    "rating": {
        "type": "score",
        "instructions": "How many stars would this reviewer likely give?",
        "criteria": ["1 star", "2 stars", "3 stars", "4 stars", "5 stars"],
    },
}

wait_until_ready()
answers = ask(review, questions)

# choice: the option the model picked
sentiment = answers["sentiment"]
print(f"Sentiment: {sentiment['choice']} (confidence {sentiment['confidence']})")

# noul: the probability that the answer is yes
p_yes = answers["recommends"]["noul"]
print(f"Recommends: {'yes' if p_yes >= 0.5 else 'no'} (probability of yes {p_yes})")

# score: a position on the scale, counted from 0; legend maps positions to labels
rating = answers["rating"]
label = rating["legend"][str(round(rating["score"]))]
print(f"Rating: {label} (score {rating['score']}, confidence {rating['confidence']})")

### Clean up

When you're done, ask your agent to delete the `openjev` endpoint. With 0 active workers, an idle endpoint costs nothing, but deleting it keeps your account tidy.

## Resources

- [Getting started with GLM-5.3 with Runpod's public endpoint](https://www.runpod.io/blog/getting-started-glm-5-3-runpod-public-endpoint)
- [How to Deploy OpenJev on Runpod Serverless](https://www.runpod.io/blog/how-to-deploy-openjev-on-runpod-serverless)
- [Runpod MCP server](https://github.com/runpod/runpod-mcp)
- [Public Endpoints docs](https://docs.runpod.io/public-endpoints/overview)
- [Serverless load balancing docs](https://docs.runpod.io/serverless/load-balancing/overview)
- [Serverless pricing](https://docs.runpod.io/serverless/pricing)

The OpenJev model weights are licensed under CC BY-NC 4.0, which allows non-commercial use with attribution.